# MAI-Voice-2.1: give A Bit of AI a voice

Create the opening of **A Bit of AI**, a podcast hosted by Henk and Amy.
The pattern throughout is **write SSML → post it → play the result**.
Voice names, model choices, styles, and speaker changes are visible in the code.

[Model information](information.md) · [Setup instructions](demo-instructions.md)

`post_ssml(...)` handles the endpoint, authentication headers, HTTP request,
validation, and response decoding. It sends the exact SSML you supply and returns
an `AudioResult`. After success it reports **spoken characters, estimated cost,
full request time, and audio duration**. We call `Audio(result.audio)` explicitly
to create a player; press Play to listen. Replaying the player makes no API call.

Configure your key, then run individual synthesis cells to generate audio.
**Running a synthesis cell sends paid requests immediately**, including on reruns.
Run the optional cloning steps individually: they upload recordings and create
a stored personal voice. Do not use Run All for that workflow. There are no automatic retries or model/voice fallbacks.

Both models are **public preview**, without an SLA. All generated speech is
synthetic. Grant and Harper are prebuilt stand-ins, not the hosts' real voices.
Live model/region availability and output quality need checking on your resource.

In [ ]:
from html import escape
import json
from pathlib import Path

from IPython.display import Audio, display
from voice_demo import (
    Settings, SpeechClient, PersonalVoiceClient, Approval, DemoError,
    DISPUTED_REGIONS, recording,
)

ROOT = Path.cwd()
if not (ROOT / "scripts.json").is_file():
    raise RuntimeError("Start this notebook with MAI-Voice-2.1 as its working directory.")
SCRIPTS = json.loads((ROOT / "scripts.json").read_text(encoding="utf-8"))

In [ ]:
# Credentials come from this folder's .env or the process environment.
settings = Settings.load(ROOT)
speech = SpeechClient(settings)
post_ssml = speech.post_ssml

print("Voice region:", settings.region or "(not configured)")
print("Voice key:", "configured (hidden)" if settings.key else "missing")
if settings.region in DISPUTED_REGIONS:
    print("Warning: this region has conflicting MAI availability documentation.")
print("After changing .env, rerun this configuration cell. No Azure calls made.")

## 1. Give the welcome a voice

Begin with Amy's introduction. The `<voice>` name selects **Harper** and
**MAI-Voice-2.1**. Edit the script or voice name directly. `escape(...)` keeps
plain text such as `AI & agents` from being interpreted as XML markup.

In [ ]:
welcome_text = "Hi everyone, and welcome to a new episode of A Bit of AI! I'm Amy, and I'm so glad you're joining us."

welcome_ssml = f"""
<speak version="1.0"
       xmlns="http://www.w3.org/2001/10/synthesis"
       xml:lang="en-US">
  <voice name="en-US-Harper:MAI-Voice-2.1">
    {escape(welcome_text)}
  </voice>
</speak>
"""

In [ ]:
result = post_ssml(welcome_ssml)
display(Audio(data=result.audio, autoplay=False))

## 2. Direct the delivery

Give Amy's quick-fire teaser three deliveries using the **same words and voice**.
`mstts:express-as` is the change to watch. Neutral, excited, and whispering
are documented for English Harper; they are not interchangeable across voices.

Running this cell makes **three requests**. Listen for the actual effect:
service success alone does not prove that a style was applied.

In [ ]:
teaser_text = SCRIPTS["dialogue"][11]["text"]

for style in ("neutral", "excited", "whispering"):
    teaser_ssml = f"""
    <speak version="1.0"
           xmlns="http://www.w3.org/2001/10/synthesis"
           xmlns:mstts="http://www.w3.org/2001/mstts"
           xml:lang="en-US">
      <voice name="en-US-Harper:MAI-Voice-2.1">
        <mstts:express-as style="{style}">{escape(teaser_text)}</mstts:express-as>
      </voice>
    </speak>
    """
    print(style)
    result = post_ssml(teaser_ssml)
    display(Audio(data=result.audio, autoplay=False))

## 3. Welcome listeners in English and Dutch

Use Grant for matching English/Dutch welcomes, then audition Sander with
`friendlycheerful` delivery. The Dutch welcome below is a prepared translation
of the English string beside it; **TTS does not translate**.

The Sander example deliberately changes casting and delivery. It is not a
controlled model comparison. Running the cell makes **three requests**.

In [ ]:
henk_en = "Hello everyone, and welcome to a new episode of A Bit of AI! I'm Henk Boelman, and I'm glad you're back."
henk_nl = "Hallo allemaal, en welkom bij een nieuwe aflevering van A Bit of AI! Ik ben Henk Boelman, en ik ben blij dat je er weer bij bent."

editions = [
    ("en-US", "en-US-Grant", "neutral", henk_en),
    ("nl-NL", "nl-NL-Grant", "neutral", henk_nl),
    ("nl-NL", "nl-NL-Sander", "friendlycheerful", henk_nl),
]

for locale, voice, style, text in editions:
    language_ssml = f"""
    <speak version="1.0"
           xmlns="http://www.w3.org/2001/10/synthesis"
           xmlns:mstts="http://www.w3.org/2001/mstts"
           xml:lang="{locale}">
      <voice name="{voice}:MAI-Voice-2.1">
        <mstts:express-as style="{style}">{escape(text)}</mstts:express-as>
      </voice>
    </speak>
    """
    print(voice, style)
    result = post_ssml(language_ssml)
    display(Audio(data=result.audio, autoplay=False))

## 4. Compare MAI-Voice-2.1 with Flash

Only the model suffix changes. Keep Harper, text, output format, and region fixed.
After each of these **two requests**, compare the audio and completion statistics.

The timer measures the **full HTTP request through completed audio download**,
not inference, time to first audio, or browser playback latency. This is not
live streaming or a formal benchmark; reverse the order and repeat manually
to explore variation. Published rates are **$22/million characters** for 2.1
and **$15/million** for Flash. The reported estimate counts spoken text only.

In [ ]:
for model in ("MAI-Voice-2.1", "MAI-Voice-2.1-Flash"):
    comparison_ssml = f"""
    <speak version="1.0"
           xmlns="http://www.w3.org/2001/10/synthesis"
           xml:lang="en-US">
      <voice name="en-US-Harper:{model}">{escape(welcome_text)}</voice>
    </speak>
    """
    result = post_ssml(comparison_ssml)
    display(Audio(data=result.audio, autoplay=False))

## 5. Bring Henk and Amy together

Render the supplied opening with **Henk = Grant** and **Amy = Harper**.
The complete original scripts remain in `scripts.json`. Speaker labels choose
the voice but are not spoken. Each `<voice>` below is one turn; the complete
prebuilt dialogue is sent in **one request**.

Listen for handoffs, the ellipses, names, and speaker consistency. The markup
is visible rather than hidden inside a dialogue helper.

In [ ]:
casting = {
    "Henk": "en-US-Grant:MAI-Voice-2.1",
    "Amy": "en-US-Harper:MAI-Voice-2.1",
}

voice_elements = "\n".join(
    f'<voice name="{casting[turn["speaker"]]}">{escape(turn["text"])}</voice>'
    for turn in SCRIPTS["dialogue"]
)

podcast_ssml = f"""
<speak version="1.0"
       xmlns="http://www.w3.org/2001/10/synthesis"
       xml:lang="en-US">
  {voice_elements}
</speak>
"""
print(podcast_ssml)

In [ ]:
result = post_ssml(podcast_ssml)
display(Audio(data=result.audio, autoplay=False))

## 6. Clone your own voice from prerecorded audio

Use Henk's own voice for the podcast welcome. There are no forms:
follow the code cells below, one step at a time.

**West Europe (`westeurope`)** is recommended; **Sweden Central
(`swedencentral`)** is another supported option. North Europe is not listed
for personal voice. Your key, region, and custom-subdomain endpoint must match.

You need [Microsoft Limited Access approval](https://aka.ms/customneural),
the speaker's explicit permission, and a recording of that speaker reading
the [official consent statement](https://github.com/Azure-Samples/Cognitive-Speech-TTS/blob/master/CustomVoice/script/verbal-statement-all-locales.txt).
A reference clip or podcast script does not replace recorded consent.
Approval eligibility is restricted; a key alone does not grant cloning access.

Personal voices incur daily storage charges. Consent, reference audio, and
profiles are stored, with documented security-retention exceptions.
This is not a zero-retention workflow. Skip this section if you only need prebuilt voices.

### Step 1. Choose your prerecorded audio files

Put two existing recordings in `samples/` and edit the paths below:

- `henk-reference.wav`: 5–60 seconds of your natural voice, without music or other speakers.
- `henk-consent.wav`: you reading the official consent statement, with your real name and company.

Use mono 16-bit PCM WAV at 16, 24, 44.1, or 48 kHz, up to 30 MB each.
Consent recordings must be 1–120 seconds for this demo.
This step validates both files locally and lets you listen to the reference.
It does not upload anything or record through your microphone.

**Recordings are not included in this repository.** Supply your own files,
keep them in the Git-ignored `samples/` folder, and never commit consent audio.

In [ ]:
reference_audio = ROOT / "samples" / "henk-reference.wav"
consent_audio = ROOT / "samples" / "henk-consent.wav"

recording(reference_audio, reference=True)
recording(consent_audio)
display(Audio(filename=str(reference_audio), autoplay=False))
print("Both recordings are valid locally. Nothing uploaded.")

### Step 2. Configure your personal voice

Use the exact talent and company names spoken in your consent recording.
The resource IDs below are your own labels, not the service's `speakerProfileId`.
Set each approval confirmation to `True` only when it is actually satisfied.
These are consent/access requirements, not payment switches.

Running this cell makes no requests and clears the in-memory profile.
Rerun it after changing the resource, identity, or approval details.
For an existing voice, set its `voice_id` here and jump to Step 7:
you do not need local recordings or another upload to reuse it.

`consent_locale` is `en-US` for an English consent recording. It does not
change the language of the Dutch reference clip or generated welcome.

In [ ]:
project_id = "abitofai-henk"
consent_id = "henk-consent"
voice_id = "henk-podcast"
talent_name = ""  # Enter the exact speaker name from the consent recording.
company_name = ""  # Enter the exact company name from the consent recording.
consent_locale = "en-US"

approval = Approval(
    enabled=True,
    approved=False,
    permission=False,
    storage_acknowledged=False,
)
personal = PersonalVoiceClient(settings, approval)
profile_id = None

### Step 3. Create the project

This calls Azure. Microsoft approval, speaker permission, and storage
acknowledgment must be confirmed in Step 2 before any personal-voice request.

In [ ]:
personal.create_project(project_id)
print("Project ready.")

### Step 4. Upload the recorded consent

This uploads your consent WAV. Run once, then check its status in Step 5.
Do not repeat uploads while processing.

In [ ]:
profile_id = None
personal.submit_consent(
    consent_id, project_id, talent_name, company_name, consent_locale, consent_audio
)
print("Consent submitted. Check its status in Step 5.")

### Step 5. Check consent processing

Rerun only this status cell until it says `Succeeded`.
`NotStarted` or `Running` means wait; failures raise an error.
Do not proceed to the reference upload until consent has succeeded.

In [ ]:
consent_status = personal.status("consents", consent_id)["status"]
print("Consent status:", consent_status)

### Step 6. Create the voice from your reference recording

This first verifies successful consent in the same project, then uploads
your reference WAV. Run once; profile creation is asynchronous.
Check progress using Step 7 instead of repeating this upload.

In [ ]:
profile_id = None
personal.submit_voice(voice_id, project_id, consent_id, reference_audio)
print("Reference submitted. Load the profile in Step 7 when ready.")

### Step 7. Load the ready voice profile

This checks both the voice and its linked consent. If still processing,
it raises a clear error; wait and rerun this cell without re-uploading.
The profile ID stays in memory and is not printed.

In [ ]:
profile_id = None
profile_id = personal.load_profile(voice_id)
print("Your consented voice is ready.")

### Step 8. Give the podcast welcome your voice

Edit the text, send visible SSML, and play the result: **one synthesis request**.
`mstts:ttsembedding` selects your verified profile. Language selection does
not translate text. Personal-voice style support is not assumed here.
Do not print the constructed SSML because it contains the profile ID.
Characters, estimated cost, request time, and audio duration appear after success.

In [ ]:
if not profile_id:
    raise DemoError("Load your successful consented profile in Step 7 first.")

personal_text = "Hallo allemaal, en welkom bij een nieuwe aflevering van A Bit of AI! Ik ben Henk Boelman, en ik ben blij dat je er weer bij bent."
personal_ssml = f"""
<speak version="1.0"
       xmlns="http://www.w3.org/2001/10/synthesis"
       xmlns:mstts="http://www.w3.org/2001/mstts"
       xml:lang="nl-NL">
  <voice name="MAI-Voice-2.1">
    <mstts:ttsembedding speakerProfileId="{profile_id}">{escape(personal_text)}</mstts:ttsembedding>
  </voice>
</speak>
"""
result = personal.post_ssml(personal_ssml)
display(Audio(data=result.audio, autoplay=False))

## What we produced

A playable podcast opening, with editable SSML showing voice selection,
styles, multilingual delivery, model comparison, and speaker changes.
The optional cloning steps give a short welcome your own consented voice.

The five core demos make **10 synthesis requests**; the personal welcome
adds **one**, plus separate profile-management requests and storage charges.
Rerunning POST cells sends new paid requests. Replay existing players
without another call. Estimates count spoken text only; billable SSML,
character weighting, storage, and subscription rates can differ.

### Cleanup when you are finished

Closing the notebook does not delete your stored voice or stop its storage
charges. To delete it, copy this into a code cell and replace the confirmation
with the exact `voice_id` you configured:

```python
personal.delete_voice(voice_id, confirmation="TYPE THE EXACT VOICE ID HERE")
profile_id = None
```

Consent/project cleanup is separate. Deletion does not override documented
service retention exceptions. Audio outputs contain generated or reference
audio: clear all outputs and any saved widget state before sharing.
Never commit keys, recordings, profile IDs, or personal audio.

[Setup and troubleshooting](demo-instructions.md) ·
[Detailed model information](information.md) ·
[Official MAI voices guide](https://learn.microsoft.com/en-us/azure/ai-services/speech-service/mai-voices)